In [15]:
import re
import numpy as np
import pandas as pd

In [33]:
cd C:\Users\tr432\Downloads\Metabolites_new\Serum_POS\mean

C:\Users\tr432\Downloads\Metabolites_new\Serum_POS\mean


In [34]:
df = pd.read_excel("../Compounds_rev.xlsx")
df.shape

(1926, 67)

In [35]:
# FeatureID 포맷
MZ_DIGITS = None   #Edit: ex) 4 / 또는 None이면 raw
RT_DIGITS = None   #Edit: ex) 3 / 또는 None이면 raw

df["m/z"]     = pd.to_numeric(df["m/z"], errors="coerce")
df["RT [min]"]= pd.to_numeric(df["RT [min]"], errors="coerce")

def fmt_num(x, digits):
    if pd.isna(x):
        return "NA"
    if digits is None:
        # raw: 불필요한 0 제거 + 과학표기 최소화
        return np.format_float_positional(x, trim='-')
    return f"{x:.{digits}f}"

df["FeatureID"] = (
    df["m/z"].map(lambda x: fmt_num(x, MZ_DIGITS)) +
    "-" +
    df["RT [min]"].map(lambda x: fmt_num(x, RT_DIGITS))
)

print(df.shape)
df.head(3)

(1926, 68)


,Tags,Checked,Name,Formula,Annot. Source: Predicted Compositions,Annot. Source: mzCloud Search,Annot. Source: ChemSpider Search,Annot. DeltaMass [ppm],Calc. MW,m/z,...,Peak Rating: QC1.raw (F13),Peak Rating: QC2.raw (F14),Peak Rating: QC3.raw (F15),Peak Rating: Vnam-1.raw (F16),Peak Rating: Vnam-2.raw (F17),Peak Rating: Vnam-3.raw (F18),Peak Rating: Vnam-4.raw (F19),Peak Rating: Vnam-5.raw (F20),Peak Rating: Vnam-6.raw (F21),FeatureID
0,NaN,False,spisulosine,C18 H39 N O,Full match,No results,Full match,0.28,285.30324,286.31052,...,7.8,8.9,8.9,8.9,7.8,7.3,7.8,8.4,7.8,286.31052-14.933
1,NaN,False,5-Hydroxymethyl-2-furaldehyde,C6 H6 O3,Full match,Full match,Partial match,-0.22,126.03167,127.03897,...,2.9,4.0,2.9,3.5,3.5,2.9,3.5,1.8,2.9,127.03897-1.441
2,NaN,False,NaN,C16 H35 N O,Full match,No results,No results,0.10,257.27189,258.27917,...,8.9,9.5,8.4,8.4,9.5,7.8,8.9,8.4,8.9,258.27917-14.002


In [36]:
# 3) FeatureID 중복 방지(동일 mz/rt가 있으면 _dup2, _dup3...)
dup = df.groupby("FeatureID").cumcount()
df.loc[dup > 0, "FeatureID"] = df.loc[dup > 0, "FeatureID"] + "_dup" + (dup[dup > 0] + 1).astype(str)

df

,Tags,Checked,Name,Formula,Annot. Source: Predicted Compositions,Annot. Source: mzCloud Search,Annot. Source: ChemSpider Search,Annot. DeltaMass [ppm],Calc. MW,m/z,...,Peak Rating: QC1.raw (F13),Peak Rating: QC2.raw (F14),Peak Rating: QC3.raw (F15),Peak Rating: Vnam-1.raw (F16),Peak Rating: Vnam-2.raw (F17),Peak Rating: Vnam-3.raw (F18),Peak Rating: Vnam-4.raw (F19),Peak Rating: Vnam-5.raw (F20),Peak Rating: Vnam-6.raw (F21),FeatureID
0,NaN,False,spisulosine,C18 H39 N O,Full match,No results,Full match,0.28,285.30324,286.31052,...,7.8,8.9,8.9,8.9,7.8,7.3,7.8,8.4,7.8,286.31052-14.933
1,NaN,False,5-Hydroxymethyl-2-furaldehyde,C6 H6 O3,Full match,Full match,Partial match,-0.22,126.03167,127.03897,...,2.9,4.0,2.9,3.5,3.5,2.9,3.5,1.8,2.9,127.03897-1.441
2,NaN,False,NaN,C16 H35 N O,Full match,No results,No results,0.10,257.27189,258.27917,...,8.9,9.5,8.4,8.4,9.5,7.8,8.9,8.4,8.9,258.27917-14.002
3,NaN,False,Urea,C H4 N2 O,No results,Full match,Full match,0.90,60.03242,61.03969,...,8.9,9.5,8.9,9.5,9.5,9.5,9.5,8.9,9.5,61.03969-1.062
4,NaN,False,Lysolecithin,C24 H50 N O7 P,Full match,No results,Full match,0.40,495.33269,496.33995,...,8.9,9.5,8.9,9.5,9.5,9.5,9.5,8.9,8.9,496.33995-14.389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1921,NaN,False,NaN,NaN,No results,No results,No results,NaN,362.26709,363.27442,...,2.2,2.2,NaN,5.4,NaN,NaN,2.7,2.7,NaN,363.27442-15.162
1922,NaN,False,NaN,NaN,No results,No results,No results,NaN,315.17327,316.18052,...,NaN,1.6,4.3,NaN,NaN,NaN,NaN,3.8,NaN,316.18052-7.669
1923,NaN,False,NaN,C H3 Cl2 O7 P3,Full match,No results,No results,3.98,289.84802,290.85530,...,NaN,6.5,NaN,7.1,6.5,NaN,NaN,NaN,NaN,290.8553-0.999
1924,NaN,False,NaN,NaN,No results,No results,No results,NaN,360.88161,361.88999,...,NaN,4.9,NaN,3.3,4.9,NaN,NaN,NaN,NaN,361.88999-0.954


In [37]:
# cols 정의 추가 (NameError 해결)
cols = df.columns.astype(str).tolist()

# 4) 컬럼 탐지
bio_cols = [c for c in cols if re.match(r'^[CV]\d+_(0w|1w|2w)$', c)]
qc_cols  = [c for c in cols if re.match(r'^QC\d+', c, re.IGNORECASE)]
bl_cols  = [c for c in cols if re.match(r'^BL\d+', c, re.IGNORECASE)]

def rename_bio(c):
    m = re.match(r'^([CV])(\d+)_', c)
    prefix = "Control" if m.group(1) == "C" else "VNAM"
    sid = str(int(m.group(2)))   # 01 -> 1
    return f"{prefix}-{sid}"

def rename_qc(c):
    m = re.match(r'^(QC)(\d+)', c, re.IGNORECASE)
    sid = str(int(m.group(2))) if m else c
    return f"QC{sid}"

def rename_bl(c):
    m = re.match(r'^(BL)(\d+)', c, re.IGNORECASE)
    sid = str(int(m.group(2))) if m else c
    return f"Blank{sid}"

def add_group_row_and_save(feature_ids, X, out_csv):
    # 인덱스 정렬 이슈 방지: 둘 다 0..n-1로 맞춤
    X2  = X.reset_index(drop=True)
    fid = pd.Series(feature_ids).reset_index(drop=True)

    sample_names = list(X2.columns)

    def to_group(s):
        if s.startswith("C"):     return "Control"
        if s.startswith("V"):     return "VNAM"
        if s.startswith("QC"):    return "QC"
        if s.startswith("Blank"): return "BLANK"
        return s.split("-")[0]

    group_row = [to_group(s) for s in sample_names]

    out = pd.DataFrame({"Label": fid})
    out = pd.concat([out, X2], axis=1)

    group_df = pd.DataFrame([["Group"] + group_row], columns=["Label"] + sample_names)
    out2 = pd.concat([group_df, out], ignore_index=True)

    # 안전장치(원하면)
    # assert out2.loc[1:, "Label"].isna().sum() == 0

    out2.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[saved]", out_csv, "shape=", out2.shape)

# --- 주차별 저장 ---
INCLUDE_QC_BLANK = True  #Edit: MA에서 QC/Blank 기반 필터링하려면 True

for wk in ["0w", "1w", "2w"]:
    cols_wk = [c for c in bio_cols if c.endswith(f"_{wk}")]
    if len(cols_wk) == 0:
        print(f"[{wk}] bio 샘플 없음 -> skip")
        continue

    use_cols = cols_wk.copy()
    if INCLUDE_QC_BLANK:
        use_cols += qc_cols + bl_cols

    X = df[use_cols].apply(pd.to_numeric, errors="coerce").copy()

    # rename columns
    new_cols = []
    for c in use_cols:
    #    if c in cols_wk:
    #        new_cols.append(rename_bio(c))
        if c in qc_cols:
            new_cols.append(rename_qc(c))
        elif c in bl_cols:
            new_cols.append(rename_bl(c))
        else:
            new_cols.append(c)
    X.columns = new_cols

    # --- Sample/Blank 필터 (물질별) ---
    # # ratio = (Sample 통계치) / (Blank 통계치)
    # # S2B_STAT: "mean" 또는 "median"
    # # S2B_INCLUSIVE=True  -> ratio <= threshold 제거  (즉, 5 이하 제거)
    # # S2B_INCLUSIVE=False -> ratio <  threshold 제거  (즉, 5 미만만 제거)
    S2B_THRESHOLD = 5.0  #Edit
    S2B_INCLUSIVE = True  #Edit
    S2B_STAT = "mean"   # "mean" or "median"  #Edit
    
    sample_cols = [c for c in X.columns if str(c).startswith(("C", "V"))]
    blank_cols  = [c for c in X.columns if str(c).startswith("Blank")]
    
    if len(sample_cols) > 0 and len(blank_cols) > 0:
        if S2B_STAT == "median":
            sample_stat = X[sample_cols].median(axis=1, skipna=True).fillna(0.0)
            blank_stat  = X[blank_cols].median(axis=1, skipna=True).fillna(0.0)
        else:
            sample_stat = X[sample_cols].mean(axis=1, skipna=True).fillna(0.0)
            blank_stat  = X[blank_cols].mean(axis=1, skipna=True).fillna(0.0)
            
        s2b = sample_stat / (blank_stat)
            
        keep = (s2b > S2B_THRESHOLD) if S2B_INCLUSIVE else (s2b >= S2B_THRESHOLD)
        n_before = X.shape[0]
        X = X.loc[keep].copy()
        print(f"[{wk}] Sample/Blank filter ({S2B_STAT}) kept {X.shape[0]}/{n_before} "
              f"(removed {n_before - X.shape[0]}); "
              f"rule={'<=' if S2B_INCLUSIVE else '<'}{S2B_THRESHOLD}")
    else:
        print(f"[{wk}] Sample/Blank filter skipped (sample_cols={len(sample_cols)}, blank_cols={len(blank_cols)})")

    # (필터링 끝난 뒤) 저장용: QC/Blank 열 제거
    drop_cols = [c for c in X.columns if str(c).startswith(("Blank"))]  ## 경우에 따라 "QC"도 제외 가능
    X_save = X.drop(columns=drop_cols)

    add_group_row_and_save(df.loc[X_save.index, "FeatureID"], X_save, f"MA_{wk}.csv")  ## 이거 나중에 _rev 지워야됨_26.01.04

[0w] bio 샘플 없음 -> skip
[1w] bio 샘플 없음 -> skip
[2w] Sample/Blank filter (mean) kept 1268/1926 (removed 658); rule=<=5.0
[saved] MA_2w.csv shape= (1269, 19)


In [45]:
import re
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

In [105]:
cd C:\Users\tr432\Downloads\Metabolites_new\Serum_POS\mean

C:\Users\tr432\Downloads\Metabolites_new\Serum_POS\mean


In [106]:
MA_PATH   = r"data_normalized_2w.csv"
COMP_PATH = r"../Compounds_rev.xlsx"
OUT_XLSX  = r"MA_stats+annot_2w.xlsx"

In [107]:
# -----------------------------
# 1) MA_2w.csv 읽기 + 그룹행 파싱
# -----------------------------
ma = pd.read_csv(MA_PATH)

# 첫 컬럼명을 Label로 통일
ma = ma.rename(columns={ma.columns[0]: "Label"})

# 그룹행 찾기 (Group 또는 Label)
s = ma["Label"].astype(str).str.strip().str.lower()
grp_idx = ma.index[s.isin(["group", "label"])][0]

# sample -> group
grp_row = ma.loc[grp_idx, ma.columns[1:]].astype(str).str.strip()

def norm_group(g):
    u = g.upper()
    return {"CONTROL":"Control","CTRL":"Control",
            "VNAM":"VNAM","V":"VNAM",
            "QC":"QC",
            "BLANK":"BLANK","BL":"BLANK"}.get(u, g)

sample2group = {c: norm_group(grp_row[c]) for c in grp_row.index}
ctrl_cols = [c for c,g in sample2group.items() if g=="Control"]
vnam_cols = [c for c,g in sample2group.items() if g=="VNAM"]

print("grp_idx:", grp_idx, "Control:", len(ctrl_cols), "VNAM:", len(vnam_cols))
print(ma.head(3))

grp_idx: 0 Control: 9 VNAM: 6
              Label    C03_2w    C04_2w    C05_2w    C06_2w    C07_2w  \
0             Label   Control   Control   Control   Control   Control   
1    61.03969-1.062  9.000872  9.121817  8.998766  9.150688  8.976087   
2  496.33995-14.389  9.202780  9.294801  9.223403  9.363768  9.147197   

     C08_2w    C09_2w    C10_2w    C11_2w       QC1       QC2       QC3  \
0   Control   Control   Control   Control        QC        QC        QC   
1  9.158848  9.135198  9.105664  9.146613  9.076547  9.072518  9.027376   
2  9.304696  9.273235  9.341150  9.325066  9.236528  9.150021  9.195136   

     V05_2w    V06_2w    V07_2w    V08_2w    V09_2w    V10_2w  
0      VNAM      VNAM      VNAM      VNAM      VNAM      VNAM  
1  9.064078  9.173287  9.318057  9.089451  8.943161  9.072540  
2  9.148001  9.151592  8.800738  8.968262  9.101511  9.241630  


In [108]:
# ma는 이미 읽혀있다고 가정 (MA_PATH에서 읽은 df)
ma = pd.read_csv(MA_PATH)

# 첫 컬럼명 통일
ma = ma.rename(columns={ma.columns[0]: "Label"})

# 그룹행(첫 행) 찾기: 'Group' 또는 'Label'
s = ma["Label"].astype(str).str.strip().str.lower()
grp_idx = ma.index[s.isin(["group", "label"])][0]   # data_normalized.csv면 보통 0

# 그룹행에서 sample->group 만들기
grp_row = ma.loc[grp_idx, ma.columns[1:]].astype(str).str.strip()

def norm_group(g):
    u = str(g).strip().upper()
    if u in ("CONTROL", "CTRL"): return "Control"
    if u == "VNAM":              return "VNAM"
    if u == "QC":                return "QC"
    if u in ("BLANK", "BL"):      return "BLANK"
    return u

sample2group = {c: norm_group(grp_row[c]) for c in grp_row.index}
ctrl_cols = [c for c,g in sample2group.items() if g == "Control"]
vnam_cols = [c for c,g in sample2group.items() if g == "VNAM"]

In [109]:
# -----------------------------
# 2) 데이터 본문 만들기 (그룹행 삭제, QC/Blank 컬럼 삭제)
# -----------------------------
dat = ma.drop(index=grp_idx).copy()

# 여기서 label_col 쓰지 말고, 확정된 'Label'을 FeatureID로 변경
dat = dat.rename(columns={"Label": "FeatureID"})

use_cols = ["FeatureID"] + ctrl_cols + vnam_cols
dat = dat[use_cols].copy()

for c in ctrl_cols + vnam_cols:
    dat[c] = pd.to_numeric(dat[c], errors="coerce")

dat

,FeatureID,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,C10_2w,C11_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w
1,61.03969-1.062,9.000872,9.121817,8.998766,9.150688,8.976087,9.158848,9.135198,9.105664,9.146613,9.064078,9.173287,9.318057,9.089451,8.943161,9.072540
2,496.33995-14.389,9.202780,9.294801,9.223403,9.363768,9.147197,9.304696,9.273235,9.341150,9.325066,9.148001,9.151592,8.800738,8.968262,9.101511,9.241630
3,172.07169-4.382,5.663735,5.878383,5.735270,5.883901,5.853907,6.174753,6.390740,5.825692,6.136602,9.119635,9.195488,6.327427,8.284308,9.107229,8.580883
4,520.34-13.811,9.127682,9.165927,9.056115,9.192668,8.954176,9.104596,9.094193,9.189793,9.206881,9.043104,8.997126,8.541896,8.765928,9.051915,9.179263
5,203.05255-1.03,8.933656,8.915398,8.985105,8.086934,8.914240,8.989631,9.145758,9.072946,9.044150,8.640508,7.377049,8.779044,8.354150,8.780215,8.808622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
947,323.88963-1.034,5.383596,5.346711,5.138171,5.537326,5.263164,5.824139,5.406682,5.489075,5.417887,5.734305,5.660718,5.232272,5.590217,5.658857,5.569196
948,458.25154-8.665,5.206155,5.416672,5.549438,5.262548,5.540509,5.585438,5.422717,5.520062,5.438671,5.716737,5.728612,5.642890,5.681869,5.587805,5.501325
949,526.32604-9.188,5.395562,5.201310,5.293900,5.148522,5.556205,5.027059,5.097941,5.131934,4.851002,5.701181,5.590203,5.708164,5.490948,5.434165,5.223636
950,500.27764-12.894,5.681457,5.084920,5.154721,5.654187,5.369701,5.214943,5.251065,5.062809,5.004113,5.682546,5.664010,5.115181,4.970919,5.496361,5.626688


In [110]:
# -----------------------------
# 3) long 포맷 + (선택) autoscaling + 유의성 검정
# -----------------------------
# NOTE:
# - autoscaling(z-score)은 feature별로 (값-평균)/표준편차 변환입니다.
# - Mann–Whitney U는 순위 기반이라, autoscaling을 해도 p-value는 (거의) 동일합니다.
#   (단, SD=0인 feature는 모두 같은 값이므로 z=0 처리)

USE_AUTOSCALE = True   # Edit: True면 p-value 계산에 z-score(autoscaling) 사용
AUTOSCALE_DDOF = 1     # Edit: 1=sample SD (MA/R의 scale과 유사), 0=population SD

long = dat.melt(id_vars="FeatureID", var_name="Sample", value_name="Intensity")
long["Group"] = long["Sample"].map(sample2group)
# QC/BLANK는 이미 컬럼에서 제거됐지만, 방어적으로 한 번 더
long = long[long["Group"].isin(["Control", "VNAM"])].dropna(subset=["Intensity"])

# --- autoscaling (feature-wise; Control+VNAM 합쳐서) ---
if USE_AUTOSCALE:
    mu = long.groupby("FeatureID")["Intensity"].transform("mean")

    def _sd(x):
        return x.std(ddof=AUTOSCALE_DDOF)

    sd = long.groupby("FeatureID")["Intensity"].transform(_sd)
    sd_safe = sd.where(np.isfinite(sd) & (sd != 0), other=1.0)

    long["Intensity_for_test"] = (long["Intensity"] - mu) / sd_safe
else:
    long["Intensity_for_test"] = long["Intensity"]

rows = []
for fid, sub in long.groupby("FeatureID", sort=False):
    # (1) 검정용 값 (autoscaling 적용)
    ctrl = sub.loc[sub["Group"]=="Control", "Intensity_for_test"].to_numpy(dtype=float)
    vnam = sub.loc[sub["Group"]=="VNAM",    "Intensity_for_test"].to_numpy(dtype=float)

    n_ctrl = int(np.isfinite(ctrl).sum())
    n_vnam = int(np.isfinite(vnam).sum())

    if n_ctrl > 0 and n_vnam > 0:
        u_stat, p = mannwhitneyu(ctrl, vnam, alternative="two-sided", method="auto")
        u_stat = float(u_stat); p = float(p)
    else:
        u_stat, p = np.nan, np.nan

    rows.append([fid, n_ctrl, n_vnam, u_stat, p])

res = pd.DataFrame(rows, columns=[
    "FeatureID", "Ctrl_n", "Vnam_n",
    "U_stat", "p-val"
])

# -----------------------------
# BH / BKY (패키지) 보정
# -----------------------------
alpha = 0.05  # Edit

p = res["p-val"].to_numpy(dtype=float)
ok = np.isfinite(p)

res["q-val_BH"]  = np.nan
res["q-val_BKY"] = np.nan

# BH
res.loc[ok, "q-val_BH"] = multipletests(p[ok], alpha=alpha, method="fdr_bh")[1]

# BKY (two-stage)
res.loc[ok, "q-val_BKY"] = multipletests(p[ok], alpha=alpha, method="fdr_tsbky")[1]

In [111]:
# -----------------------------
# 4) Compounds_rev.xlsx 기반으로
#    (1) FeatureID_key(mz-RT) 생성
#    (2) Ctrl/VNAM mean + lfc(log2FC) 계산 (RAW 기준)
#    (3) res와 merge + Name + Annotated
# -----------------------------
WK = "2w"   # "0w" / "1w" / "2w"로 바꿔서 실행
EPS = 1e-12

# (A) Compounds 로드
comp = pd.read_excel(COMP_PATH)

# (B) Compounds에서 FeatureID_key 생성(가능하면 자리수 통일 권장)
#     - MA의 FeatureID가 "mz-rt" 문자열이라면, 아래 DIGITS를 MA에서 만든 것과 동일하게 맞추세요.
def make_feature_key_from_mz_rt(df, mz_col, rt_col):
    df = df.copy()
    df[mz_col] = pd.to_numeric(df[mz_col], errors="coerce")
    df[rt_col] = pd.to_numeric(df[rt_col], errors="coerce")
    df["FeatureID_key"] = (
        df[mz_col].map(lambda x: f"{x}" if pd.notna(x) else "NA") + "-" +
        df[rt_col].map(lambda x: f"{x}" if pd.notna(x) else "NA")
    )
    return df

if "FeatureID" in comp.columns:
    # Compounds에 FeatureID가 이미 "mz-rt" 형태로 있다면 그대로 key로 사용
    # (단, 자리수 불일치로 매칭이 안 될 수 있으면 아래처럼 포맷 통일 권장)
    comp["FeatureID_key"] = comp["FeatureID"].astype(str).str.strip()
else:
    mz_col = next((c for c in ["m/z","mz","MZ","Mass","mass"] if c in comp.columns), None)
    rt_col = next((c for c in ["RT [min]","RT(min)","RT","rt","Retention Time","Retention time"] if c in comp.columns), None)
    if mz_col is None or rt_col is None:
        raise ValueError(f"Compounds_rev.xlsx에서 m/z 또는 RT 컬럼을 못 찾음. 현재 컬럼: {list(comp.columns)}")
    comp = make_feature_key_from_mz_rt(comp, mz_col=mz_col, rt_col=rt_col)

# (C) Compounds에서 해당 주차 RAW 컬럼(C#, V#) 잡고 mean/lfc 계산
cols = comp.columns.astype(str).tolist()
ctrl_cols_raw = [c for c in cols if re.match(rf"^C\d+_{WK}$", c)]
vnam_cols_raw = [c for c in cols if re.match(rf"^V\d+_{WK}$", c)]

if len(ctrl_cols_raw) == 0 or len(vnam_cols_raw) == 0:
    raise ValueError(f"[{WK}] Compounds에서 C/V 컬럼을 못 찾음. "
                     f"ctrl_cols={len(ctrl_cols_raw)}, vnam_cols={len(vnam_cols_raw)}")

comp[ctrl_cols_raw] = comp[ctrl_cols_raw].apply(pd.to_numeric, errors="coerce")
comp[vnam_cols_raw] = comp[vnam_cols_raw].apply(pd.to_numeric, errors="coerce")

comp["Ctrl_mean_raw"] = comp[ctrl_cols_raw].mean(axis=1, skipna=True)
comp["Vnam_mean_raw"] = comp[vnam_cols_raw].mean(axis=1, skipna=True)
comp["lfc_raw"] = np.log2((comp["Vnam_mean_raw"] + EPS) / (comp["Ctrl_mean_raw"] + EPS))

# (D) Name 정리 + 중복 FeatureID_key 처리(이름 있는 쪽 우선)
name_col = "Name" if "Name" in comp.columns else None
use_cols = ["FeatureID_key", "Ctrl_mean_raw", "Vnam_mean_raw", "lfc_raw"] + ([name_col] if name_col else [])

comp_use = comp[use_cols].copy()

if name_col:
    comp_use["_has_name"] = comp_use[name_col].notna() & (comp_use[name_col].astype(str).str.strip() != "")
    comp_use = (comp_use.sort_values(["FeatureID_key","_has_name"], ascending=[True, False])
                        .drop_duplicates("FeatureID_key", keep="first")
                        .drop(columns=["_has_name"]))
else:
    comp_use = comp_use.drop_duplicates("FeatureID_key", keep="first")
    comp_use["Name"] = np.nan
    name_col = "Name"

# (E) res 쪽 FeatureID_key 만들기 (MA FeatureID의 _dup 제거 + 자리수 통일 권장)
def key_from_featureid(fid):
    s = str(fid).strip()
    s = re.sub(r"_dup\d+$", "", s)
    if "-" in s:
        a, b = s.split("-", 1)
        try:
            mz = float(a); rt = float(b)
            return f"{mz:.{MZ_DIGITS}f}-{rt:.{RT_DIGITS}f}"
        except:
            return s
    return s

res["FeatureID_key"] = res["FeatureID"].map(key_from_featureid)

# (F) merge: RAW 기반 mean/lfc + Name 붙이기
out = res.merge(comp_use, on="FeatureID_key", how="left")

# lfc 컬럼명/mean 컬럼명 정리
out = out.rename(columns={
    "Ctrl_mean_raw": "Ctrl_mean",
    "Vnam_mean_raw": "Vnam_mean",
    "lfc_raw": "lfc",
})

out["Annotated"] = np.where(out[name_col].notna() & (out[name_col].astype(str).str.strip() != ""), "o", "x")

# key 컬럼은 최종에 필요 없으면 제거
out = out.drop(columns=["FeatureID_key"])

out

,FeatureID,Ctrl_n,Vnam_n,U_stat,p-val,q-val_BH,q-val_BKY,Ctrl_mean,Vnam_mean,lfc,Name,Annotated
0,61.03969-1.062,9,6,27.0,1.000000,1.000000,0.711041,9.659259e+08,1.438504e+09,0.574585,Urea,o
1,496.33995-14.389,9,6,49.0,0.007592,0.027350,0.019447,1.488395e+09,1.335261e+09,-0.156635,Lysolecithin,o
2,172.07169-4.382,9,6,1.0,0.000799,0.012882,0.009160,7.646520e+05,8.884390e+08,10.182254,Metronidazole,o
3,520.34-13.811,9,6,45.0,0.035964,0.084868,0.060345,1.054390e+09,1.018475e+09,-0.049998,1-Linoleoyl-2-Hydroxy-sn-glycero-3-PC,o
4,203.05255-1.03,9,6,49.0,0.007592,0.027350,0.019447,7.274601e+08,4.468759e+08,-0.702994,Phenyl sulfoxide,o
...,...,...,...,...,...,...,...,...,...,...,...,...
946,323.88963-1.034,9,6,13.0,0.113487,0.191358,0.136063,2.229898e+05,4.439720e+05,0.993491,NaN,x
947,458.25154-8.665,9,6,4.0,0.004795,0.020728,0.014739,2.259919e+05,4.951888e+05,1.131708,NaN,x
948,526.32604-9.188,9,6,5.0,0.007592,0.027350,0.019447,1.454962e+05,3.947667e+05,1.440018,NaN,x
949,500.27764-12.894,9,6,20.0,0.455944,0.535974,0.381099,1.820510e+05,3.412508e+05,0.906490,NaN,x


In [112]:
# -----------------------------
# (추가) Interested 컬럼 만들기 (Name 기준)
# -----------------------------
# 카테고리별 관심물질 정의
INTEREST_DICT = {
    "Tryptophan metabolism": [
        "L-Tryptophan",
        "DL-Tryptophan",
        "N-Acetylserotonin",
        "Serotonin",
        "5-Hydroxyindole-3-acetic acid",
        "5-Hydroxy-DL-tryptophan",
        "Indole-3-acetaldehyde",
        "Tryptamine",
        "Indole-3-pyruvic acid",
        "INDOLE-3-PYRUVATE",
        "N1-Acetyl-5-methoxykynuramine",  # 얜 Formyl-N-acetyl-5-methoxykynurenamine 랑 같은 물질인지 더 찾아봐야됨
        "6-Hydroxymelatonin",
        "5-Methoxyindoleacetic acid",
        "4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid",
        "4-(2-Aminophenyl)-2,4-dioxobutanoic acid",
        "Indole-3-acetic acid", # --> 여기까지가 MetaboAnalyst 6.0 [Pathway Analysis] 상에서 매칭시킨 물질
        "3-hydroxy-3-methyloxindole",
        "Indole",
        "Oxindole-3-acetic acid",
        "Tryptophol",  # Indole-3-ethanol 이랑 같은 물질 (from KEGG)
        "Kynurenic acid",
        "6-Hydroxykynurenic acid",
        "3-hydroxy-D-kynurenine"
    ],
    "Glycerophospholipid metabolism": [
        "PC(18:3(9Z,12Z,15Z)/18:2(9Z,12Z))",
        "PC(18:1/0:0)",
        "LysoPC(18:3(9Z,12Z,15Z))",
        "LysoPC(P-18:0)",
        "LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))",
        "L-α-lysophosphatidylcholin",
        "lysophosphatidylethanolamine (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)",  # 2-Acyl-sn-glycero-3-phosphoethanolamine 이랑 같은 물질..? (from KEGG)
        "lysophosphatidylcholine 14:1(9Z)/0:0",  # 2-Acyl-sn-glycero-3-phosphoethanolamine 이랑 같은 물질..? (from KEGG)
        "Lysolecithin",
        "Choline",
        "Acetylcholine",
    ],
    "TCA cycle": [
        "2-Oxoglutaric acid",
        "Succinic acid",
        "L-(-)-Malic acid",
        "(Z)-Aconitic Acid",
        "trans-Aconitic acid",
        "Citric acid"
    ],
    "SCFAs": [
        "Acetic acid",
        "Acetate",
        "Propionic acid",
        "Propionate",
        "Butyric acid",
        "Butyrate"
    ],
#    "Amino acids": [
#        "L-Proline",
#    ],
    "Interested": [
        "Methyl indole-3-acetate",
        "dl-3-Indolelactic acid",
        "Indole-3-lactic acid"
    ],
}

def norm_key(s):
    return str(s).strip().lower()

# out에서 매칭 기준 컬럼: 기본은 Name (Step4에서 name_col="Name"으로 설정됨)
FEAT_COL = "Name"
if FEAT_COL not in out.columns:
    raise ValueError(f"out에 '{FEAT_COL}' 컬럼이 없습니다. 현재 컬럼: {list(out.columns)}")

# feature -> categories(set)
interest_map = {}
for cat, feats in INTEREST_DICT.items():
    for f in feats:
        interest_map.setdefault(norm_key(f), set()).add(cat)

# Interested 값 만들기
def to_interested(x):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    cats = interest_map.get(norm_key(x), set())
    return "; ".join(sorted(cats)) if cats else ""

out["Interested"] = out[FEAT_COL].apply(to_interested)

# (선택) 매칭 요약 출력
avail = set(out[FEAT_COL].dropna().astype(str).map(norm_key))
req = set()
for cat, feats in INTEREST_DICT.items():
    for f in feats:
        req.add(norm_key(f))

matched = sorted(req & avail)
missing = sorted(req - avail)


print("\n[Interested match summary]")
print("Requested:", len(req))
print("Matched  :", len(matched))
print("Missing  :", len(missing))

# --- Matched 출력 ---
print("\n[Matched (found in out['Name'])]")
if matched:
    for k in matched:
        print(" -", k)
else:
    print(" (none)")

# --- Missing 출력 ---
print("\n[Missing (NOT found in out['Name'])]")
if missing:
    for k in missing:
        print(" -", k)
else:
    print(" (none)")


[Interested match summary]
Requested: 49
Matched  : 12
Missing  : 37

[Matched (found in out['Name'])]
 - choline
 - citric acid
 - dl-tryptophan
 - indole-3-acetic acid
 - l-tryptophan
 - lysolecithin
 - lysopc(18:3(9z,12z,15z))
 - lysopc(20:5(5z,8z,11z,14z,17z))
 - lysophosphatidylethanolamine (22:6(4z,7z,10z,13z,16z,19z)/0:0)
 - methyl indole-3-acetate
 - pc(18:1/0:0)
 - tryptophol

[Missing (NOT found in out['Name'])]
 - (z)-aconitic acid
 - 2-oxoglutaric acid
 - 3-hydroxy-3-methyloxindole
 - 3-hydroxy-d-kynurenine
 - 4-(2-amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid
 - 4-(2-aminophenyl)-2,4-dioxobutanoic acid
 - 5-hydroxy-dl-tryptophan
 - 5-hydroxyindole-3-acetic acid
 - 5-methoxyindoleacetic acid
 - 6-hydroxykynurenic acid
 - 6-hydroxymelatonin
 - acetate
 - acetic acid
 - acetylcholine
 - butyrate
 - butyric acid
 - dl-3-indolelactic acid
 - indole
 - indole-3-acetaldehyde
 - indole-3-lactic acid
 - indole-3-pyruvate
 - indole-3-pyruvic acid
 - kynurenic acid
 - l-(-)-malic ac

In [113]:
# -----------------------------
# 5) 저장 (통계결과 + (원하면) long)
# -----------------------------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    out.to_excel(w, index=False, sheet_name="stats_all")
    # 필요하면 다른 시트도 추가 가능
    # long.to_excel(w, index=False, sheet_name="long_for_plot")

print("Saved:", OUT_XLSX, "shape=", out.shape)

Saved: MA_stats+annot_2w.xlsx shape= (951, 13)
